In [ ]:
import pickle
import json
import copy
from pprint import pprint as pp
import io
%matplotlib inline

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.base import is_classifier, is_regressor
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, KBinsDiscretizer, PowerTransformer, QuantileTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, classification_report
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, BaseCrossValidator, GroupShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from xgboost import XGBRegressor, XGBClassifier

# Set figsize global params
plt.rcParams['figure.figsize'] = (12,5)
plt.rcParams['axes.labelsize'] = 10

TREE_MODELS = (
    RandomForestRegressor,
    XGBRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    XGBClassifier,
    # LGBMRegressor,
    # LGBMClassifier,
    # CatBoostRegressor,
    # CatBoostClassifier,
)

RANDOM_STATE=42

## Preprocessing Pipeline

In [ ]:
def make_preprocessor(gaussian_cols=None, skewed_cols=None,
                      multimodal_cols=None, categorical_cols=None,
                      model_type="tree"):
    """
    Builds a universal preprocessing pipeline with optional scaling.
    Safety handles missing, empty, or completely absent numeric/categorical feature sets.
    """
    transformers = []

    if gaussian_cols:
        transformers.append(
            ('gaussian',
             StandardScaler(),
             gaussian_cols))

    if skewed_cols:
        transformers.append(
            ('skewed',
             PowerTransformer(method='yeo-johnson'),
             skewed_cols))

    if multimodal_cols:
        if model_type=="tree":
            transformers.append(
                ('multimodal',
                 'passthrough',
                 multimodal_cols))
        else:
            transformers.append(
                ('multimodal',
                 QuantileTransformer(output_distribution='normal', n_quantiles=500),
                 multimodal_cols))

    if categorical_cols:
        transformers.append(
            ('cat',
             OneHotEncoder(handle_unknown='ignore'),
             categorical_cols))

    return ColumnTransformer(transformers, remainder='passthrough')


def build_pipeline(preprocessor, model, transform_target=False):
    """
    Builds an end-to-end scikit-learn Pipeline.

    Parameters:
    -----------
    preprocessor : transformer object
        The ColumnTransformer handling feature scaling/encoding.
    model : estimator object
        The core machine learning model (Classifier or Regressor).
    transform_target : bool, default=False
        If True and the model is a regressor, applies a log-transform to 'y'.
    """

    # Verify if the model is actually a regressor before trying to transform 'y'
    # is_regressor = getattr(model, "_estimator_type", None) == "regressor"

    if transform_target and is_regressor(model):
        # wrap the core model with the target transformation layer
        final_estimator = TransformedTargetRegressor(
            regressor=model,
            # func=np.log1p,
            # inverse_func=np.expm1
            transformer=PowerTransformer(method='yeo-johnson')  # because we have negative values
        )
    else:
        final_estimator = model


    pipe = Pipeline([
        ('preprocess', preprocessor),
        ('model', final_estimator)
    ])



    return pipe


## Basic Model Training Helper

In [ ]:
def evaluate_model(model, X_test, y_test):
    """
    All-in-one evaluator
    Auto-detects task type (Classification vs Regression) and reutnrs appropriate metrics
    """

    y_pred = model.predict(X_test)

    # --- Robust Task Auto-Detection ---
    # Start by checking the top-level estimator
    # estimator_type = getattr(model, "_estimator_type", None)


    # If it's a pipeline, inspect the final execution step
    # if estimator_type is not None and hasattr(model, "steps"):
    #     final_step_model = model.steps[-1][1]
    #     estimator_type = getattr(final_step_model, "_estimator_type", None)


    # Method 1: Check if the model explicitely declares itself as a classifier
    # is_classifier = (estimator_type == "classifier")

    # ---- CLASSIFIER WORKFLOW ----
    if is_classifier(model):
        # Get the actual model object instsance from the final pipeline step
        final_estimator = model.steps[-1][1] if hasattr(model, "steps") else model

        # If classifier supports predict_proba
        if hasattr(final_estimator, "predict_proba"):
            y_proba = model.predict_proba(X_test)[:, 1]
        elif hasattr(final_estimator, "decision_function"):
            y_proba = model.decision_function(X_test)
        else:
            # Fallbacl to hard predictions if no continuous score exists
            y_proba = y_pred

        # Extract classes safely (handles pipelines or raw estimators)
        labels = getattr(model, "classes_", None)
        if labels is None and hasattr(model, "steps"):
            labels = getattr(model.steps[-1][1], "classes_", None)

        metrics = {
            "task_type": "classification",
            "test_auc": roc_auc_score(y_test, y_proba),
            "confusion_matrix": confusion_matrix(y_test, y_pred, labels=labels),
            "report": classification_report(y_test, y_pred, output_dict=False, zero_division=0),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
        }

        return metrics

    # ---- REGRESSION WORKFLOW ----
    else:
        # Calculate standard continuous metrics
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Calculate Adjusted R² of X_test shape is accessible
        n = X_test.shape[0]
        p = X_test.shape[1] if hasattr(X_test, "shape") else 1
        adj_r2 = 1 - (1 - r2) * (n-1) / (n - p - 1) if n > (p + 1) else r2

        metrics = {
            "task_type": "regression",
            "mse": mse,
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
            "adjusted_r2": adj_r2,
            "y_test": y_test,  # added for plotting
            "y_pred": y_pred   # added for plotting
        }

        return metrics

## GridSearchCV Wrapper

In [ ]:
def run_grid_search(pipe, param_grid, X_train, y_train, scoring, cv=5, groups=None, **kwargs):
    """
    Runs GridSearchCV on a given pipeline
    """
    grid = GridSearchCV(pipe, param_grid,
                        scoring=scoring,
                        cv=cv,
                        n_jobs=-1,
                        verbose=1
                       )

    # Base argument for fit
    fit_params = {}
    if isinstance(cv, BaseCrossValidator) and "Group" in type(cv).__name__:
        if groups is None:
            raise ValueError(
                f"The cross-validation splitter '{type(cv).__name__}' requires"
                "the 'groups' parameter, but 'groups=None' was provided."
            )
        fit_params["groups"] = groups

    grid.fit(X_train, y_train, **fit_params)

    return grid

## Multi-models Loop

In [ ]:
def run_multiple_models(models, grids, X_train, y_train, X_test, y_test, scoring, transform_target=False, **kwargs):
    """
    Runs grid search for multiple models and stores the results.
    """

    results = {}

    for label, model in models.items():
        print(f"\n===== Training {label} =====\n")

        if isinstance(model, TREE_MODELS):
            preprocessor = make_preprocessor(
                gaussian_cols=gaussian_cols, skewed_cols=skewed_cols,
                multimodal_cols=multimodal_cols, model_type="tree")
        else:
            preprocessor = make_preprocessor(
                gaussian_cols=gaussian_cols, skewed_cols=skewed_cols,
                multimodal_cols=multimodal_cols, model_type="other")

        pipe = build_pipeline(preprocessor, model)
        param_grid = grids[label]

        # Dynamically inject 'regressor__' if the target wrapper is active
        # if transform_target and getattr(model, "_estimator_type", None) == "regressor":
        if is_regressor(model):
            param_grid = {k.replace('model__', 'model__regressor__'): v for k, v in grids[label].items()}
        else:
            pram_grid = grids[label]

        grid = run_grid_search(pipe, param_grid, X_train, y_train, scoring=scoring, **kwargs)

        # Best estimator
        best_model = grid.best_estimator_

        # Evaluate
        metrics = evaluate_model(best_model, X_test, y_test)

        results[label] = {
            "best_model": best_model,
            "best_prams": grid.best_params_,
            "cv_best_score": grid.best_score_,
            "test_metrics": metrics,
            "grid": grid
        }

        cm = metrics.get('confusion_matrix', None)

        display_model_results(label, grid, metrics, cm, best_model)


    return results

## Models Librairy

In [ ]:

all_models = {
    # ---- Linear Models ----
    "lin_reg": LinearRegression(),
    # "log_reg": LogisticRegression(max_iter=500),

    # ---- Decision Trees ----
    # "dt_reg": DecisionTreeRegressor(),
    # "dt_clf": DecisionTreeClassifier(),

    # ---- Random Forests ----
    "rf_reg": RandomForestRegressor(random_state=RANDOM_STATE),
    "rf_clf": RandomForestClassifier(random_state=RANDOM_STATE),

    # ---- Boosting -----
    "gb_clf": GradientBoostingClassifier(random_state=RANDOM_STATE),
    # "ada_clf": AdaBoostClassifier(),
    "xgb_clf": XGBClassifier(
        random_state=RANDOM_STATE,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist"
    ),
    "xgb_reg": XGBRegressor(
        random_state=RANDOM_STATE,
        objective="reg:squarederror",
    ),

    # ---- Bagging -----
    # "bag_clf": BaggingClassifier(estimator=DecisionTreeClassifier()),

    # ---- SVM -----
    "lin_svc": LinearSVC(random_state=RANDOM_STATE),
    "svc_poly": SVC(kernel='poly', probability=True, random_state=RANDOM_STATE),
    "svc_rbf": SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    # "svr": SVR(kernel='rbf')

}

## Parameter Grids

In [ ]:
# Basic parameter grids
all_param_grids = {

    # ---- Linear Regression (no params) ----
    "lin_reg": {},

    # ---- Random Forest Regressor ----
    "rf_reg": {
        "model__n_estimators": [10, 100, 500],
        "model__max_depth": [None, 10, 20, 30],
        "model__min_samples_leaf": [1, 2, 4],
        "model__min_samples_split": [2, 5, 10],
        "model__max_features": ['sqrt', 'log2']
    },
    # ---- Random Forest Classifier ----
    "rf_clf": {
        "model__n_estimators": [10, 100, 500],
        "model__max_depth": [None, 5, 10, 20],
        "model__min_samples_split": [2, 5, 10],
        # "model__min_samples_leaf": [1, 2, 4],
        # "model__max_features": ['sqrt', 'log2'],
        "model__class_weight": ["balanced", "balanced_subsample"]
    },

    # ---- XGBoost Regressor ----
    "xgb_reg": {
        "model__n_estimators": [50, 100, 200],
        "model__max_depth": [2, 3, 4],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0]
    },
    # ---- XGBoost Classifier ----
    "xgb_clf": {
        "model__n_estimators": [50, 100, 200],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__max_depth": [3, 5, 7],
        "model__scale_pos_weight": [1, 2, 5],
        # "model__subsample": [0.8, 1.0],
        # "model__colsample_bytree": [0.8, 1.0],
        # "model__class_weight": ["balanced", "balanced_subsample"]
    },


    # ---- LinearSVC ----
    "lin_svc": {
        "model__C": [0.1, 1, 10],
        "model__loss": ["hinge", "squared_hinge"]
    },

    # ---- SVC (Poly Kernel) ----
    "svc_poly": {
        "model__C": [0.1, 1, 10],
        "model__degree": [2,3,4],
        "model__gamma": ["scale", .01, .1],
        "model__coef0": [0, 1],
        "model__class_weight": [None, "balanced"]
    },

    # ---- SVC (RBF Kernel) ----
    "svc_rbf": {
        "model__C": [0.1, 1, 10, 100],
        "model__gamma": ["scale", "auto", .1, .01, .001],
        "model__class_weight": ["balanced"]
    },

    # ---- SVR ----
    "svr": {
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", 0.01, 0.1],
        "model__epsilon": [0.1, 0.2, 0.5]
    }
}

## Visualization Helper Functions

In [ ]:
def show_shrunk(fig, width=480):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=300, bbox_inches='tight')
    plt.close(fig)
    display(Image(data=buf.getvalue(), width=width))


def threshold_curve(y_test, y_proba):

    thresholds = np.linspace(0, 1, 200)
    precisions = []
    recalls = []
    f1s = []

    # High-DPI text rendering
    #plt.rcParams["figure.dpi"] = 200


    for t in thresholds:
        y_pred_t = (y_proba >= t).astype(int)
        precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
        recalls.append(recall_score(y_test, y_pred_t))
        f1s.append(f1_score(y_test, y_pred_t))

    fig, ax = plt.subplots(figsize=(8,5), dpi=150)

    ax.plot(thresholds, precisions, label="Precision")
    ax.plot(thresholds, recalls, label="Recall")
    ax.plot(thresholds, f1s, label="F1 Score")
    ax.axvline(0.5, color="grey", linestyle="--", label="Default t=0.50")

    box = ax.get_position()

    ax.set_position([
        box.x0 +0,
        box.y0 - 0,
        box.width * 1,
        box.height * 1
    ])

    ax.legend(loc="best", fontsize=8)
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Score")
    ax.set_title("Precision–Recall–F1 vs Decision Threshold")
    #plt.tight_layout()
    #plt.show()
    show_shrunk(fig, width=600)




def display_model_results(name, grid, metrics, cm, best_model):
    """
    Display model results in a clean 2-column layout.
    Left: Text summary (Classification or Regression metrics)
    Right: Visual diagnostic (Confusion Matrix or Residuals Plot)
    """

    # High-DPI text rendering
    plt.rcParams["figure.dpi"] = 300

    fig = plt.figure(figsize=(26,4))
    gs = fig.add_gridspec(1, 2, wspace=0.15, width_ratios=[3,1])

    ax_text = fig.add_subplot(gs[0, 0])
    ax_text.axis("off")

    # --------------------------------------------------
    # BEST PARAMS (Grid or direct model params)
    # --------------------------------------------------
    if grid is None:
        # not from grid search → pull params from model
        try:
            raw_params = best_model.get_params()
            params_pretty = json.dumps(raw_params, indent=2)
        except:
            params_pretty = "N/A"
        cv_text = ""   # no CV score available
    else:
        params_pretty = json.dumps(grid.best_params_, indent=2)
        # Determine the appropriate label for the CV metric
        metric_name = "AUC (CV):  " if metrics["task_type"] == "classification" else "Score (CV):"
        cv_text = f"{metric_name}:   {grid.best_score_:.4f}\n"

    # --------------------------------------------------
    # Text block content
    # --------------------------------------------------
    # Classifier task
    if metrics["task_type"] == "classification":
        text = (
            f"============= {name} (Classification) =============\n\n"
            f"Best Params:\n{params_pretty}\n\n"
            f"{cv_text}"
            f"AUC (Test): {metrics['test_auc']:.4f}\n\n"
            f"F1 (Test):  {metrics['f1']:.4f}\n\n"
            f"Classification Report:\n{metrics['report']}"
        )
    # Regression task
    else:
        text = (
            f"============= {name} (Regression) =============\n\n"
            f"Best Params:\n{params_pretty}\n\n"
            f"{cv_text}"
            f"R² (Test):       {metrics['r2']:.4f}\n"
            f"Adjusted R²:     {metrics['adjusted_r2']:.4f}\n"
            f"RMSE (Test):     {metrics['rmse']:.4f}\n"
            f"MAE (Test):      {metrics['mae']:.4f}\n"
        )

    # Render text cleanly
    ax_text.text(
        0.0, 1.0, text,
        fontsize=9,
        va="top",
        ha="left",
        family="monospace",
        transform=ax_text.transAxes
    )


    # --------------------------------------------------
    # Right Column Visualizations
    # --------------------------------------------------
    ax_right = fig.add_subplot(gs[0, 1])

    if metrics["task_type"] == "classification":
        # Render Confusion Matrix
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=getattr(best_model, "classes_", ["0", "1"])
        )
        disp.plot(
            ax=ax_right,
            cmap="Blues",
            colorbar=False,
            values_format="d"
        )
        ax_right.set_title("Confusion Matrix", fontsize=10)

    else:
        # Render Regression Residuals / Prediction Fit Plot
        y_test_vals = metrics["y_test"]
        y_pred_vals = metrics["y_pred"]

        ax_right.scatter(y_test_vals, y_pred_vals, alpha=0.5, color="purple", edgecolors="w", s=20)

        # Draw a perfect diagonal identity reference line
        ideal_min = min(min(y_test_vals), min(y_pred_vals))
        ideal_max = max(max(y_test_vals), max(y_pred_vals))
        ax_right.plot([ideal_min, ideal_max], [ideal_min, ideal_max], 'r--', alpha=0.7, lw=1.5)

        ax_right.set_title("Actual vs. Predicted", fontsize=10)
        ax_right.set_xlabel("True Values", fontsize=8)
        ax_right.set_ylabel("Predictions", fontsize=8)
        ax_right.tick_params(labelsize=7)

    # Maintain your customized sub-axes positioning metrics
    box = ax_right.get_position()
    ax_right.set_position([
        box.x0 - .37,
        box.y0 + .45,
        box.width * 0.35,
        box.height * 0.35
    ])

    plt.show()

## Threshold tuning Helper Function

In [ ]:
def find_best_threshold(y_test, y_proba, beta=1):

    thresholds = np.linspace(0, 1, 200)

    # Initialize default values
    best_t = 0.5
    best_score = -1

    for t in thresholds:
        y_pred_t = (y_proba >= t).astype(int)
        score = fbeta_score(y_test, y_pred_t, beta=beta, zero_division=0)

        if score > best_score:
            best_score = score
            best_t = t
    return best_t, best_score




def eval_at_threshold(y_test, y_proba, threshold=0.5, beta=2):
    """
    Evaluate precision/recall/F1/Fβ at a custom decision threshold.

    Returns a dictionary of metrics:
        -threshold
        -precision
        -recall
        -f1
        -fbeta
        -confusion matrix
        -test_auc
        -classification report
    """

    # 1. Apply threshold → get binary predictions
    y_pred = (y_proba >= threshold).astype(int)

    # 2. Compute metrics
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall    = recall_score(y_test, y_pred)
    f1        = f1_score(y_test, y_pred)
    fbeta     = fbeta_score(y_test, y_pred, beta=beta, zero_division=0)


    # 3. Confusion matrix plot
    cm = confusion_matrix(y_test, y_pred)

    return {
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fbeta": fbeta,
        "confusion_matrix": cm,
        "test_auc": roc_auc_score(y_test, y_proba),
        "report": classification_report(y_test, y_pred, output_dict=False)
    }

---

# Load Dataset

In [ ]:
all_tests_v3_large_turbo = {}

# with open('./results/backup/export_all_tests_v3_large_turbo_500samples.pkl', 'rb') as f:
with open('./results/backup/rescued_all_tests_v3_large_turbo_ALL.pkl', 'rb') as f:
    all_tests_v3_large_turbo = pickle.load(f)

print(f"dataset keys: {all_tests_v3_large_turbo.keys()}")

In [ ]:
# Normalize each dataset into a list of dataframes
df_list = [pd.json_normalize(data) for data in all_tests_v3_large_turbo.values()]

# Combine them vertically and use the dictionary keys as a new columns to id them
df_tests = pd.concat(df_list, keys=all_tests_v3_large_turbo.keys()).reset_index(level=0, names="test_identifier")

In [ ]:
df_tests = pd.concat(df_list, keys=all_tests_v3_large_turbo.keys()).reset_index(level=0, names="test_identifier")
cols = [col.split(".") for col in df_tests.columns]

exclude = {"metrics", "asr_flags", "reasons"}

cleaned_cols = [
    ".".join(
        item.replace("c_analysis", "c_anly")
        .replace("n_analysis", "n_anly")
        .replace("digital_clipping_detected", "clipping_detected")
        for item in sublist
        if item not in exclude
    )
    for sublist in cols
]

test_id_map = {
    'whitenoise': 1,
    'synth-verb': 2,
    'real-verb': 3,
    'dropout': 4,
    'whitenoise+synth-verb+dropout': 5
}

df_tests["test_identifier"] = df_tests["test_identifier"].map(test_id_map)

df_tests.columns = cleaned_cols
df_tests.info()
# pprint(cleaned_cols)

In [ ]:
df_tests

In [ ]:
# == Some data engineering ==
df_tests["delta_wer"] = df_tests["n_wer"] - df_tests["c_wer"]
df_tests["delta_cer"] = df_tests["n_cer"] - df_tests["c_cer"]
df_tests["delta_crest_factor"] = df_tests["n_anly.crest_factor_db"] - df_tests["c_anly.crest_factor_db"]
df_tests["delta_mean_zcr"] = df_tests["n_anly.mean_zcr"] - df_tests["c_anly.mean_zcr"]
df_tests["delta_std_zcr"] = df_tests["n_anly.std_zcr"] - df_tests["c_anly.std_zcr"]
df_tests["delta_min_zcr"] = df_tests["n_anly.min_zcr"] - df_tests["c_anly.min_zcr"]
df_tests["delta_rms"] = df_tests["n_anly.rms_dbfs_global"] - df_tests["c_anly.rms_dbfs_global"]
df_tests.sort_values(by=["c_wer", "n_wer"], ascending=False, inplace=True)

df_tests[['c_wer', 'n_wer', 'delta_wer', 'c_cer', 'n_cer', 'delta_cer']].describe()

In [ ]:
# == generate column lists per type
num_cols = df_tests.select_dtypes(include=['number']).columns.tolist()
cat_cols = df_tests.select_dtypes(include=['str', 'object', 'category']).columns.tolist()

pp(f"numerical columns: {num_cols}")
pp(f"categorical columns: {cat_cols}")

In [ ]:
# == Look at cross-correlation between numerical variables ==
cols_to_correlate = [
    'delta_wer',
    # 'delta_cer',
    'n_anly.crest_factor_db',
    'n_anly.mean_zcr',
    'n_anly.rms_dbfs_global',
    'c_anly.mean_zcr',
    'c_anly.rms_dbfs_global',
    'delta_crest_factor',
    'delta_mean_zcr',
    'delta_std_zcr',
    'delta_min_zcr',
    'delta_rms']

corr_df = df_tests[cols_to_correlate].dropna()
corr_matrix = corr_df.corr(method='spearman')

sns.heatmap(corr_matrix,
            annot=True,
            cmap='coolwarm',
            fmt='.2f',
            vmin=-1, vmax=1)
plt.title("Correlation: Audio Metrics vs WER Degradation")
plt.show()

In [ ]:
df_tests[cols_to_correlate].hist(bins=100, figsize=(12,8))

In [ ]:
cols = ['delta_wer', 'n_anly.mean_zcr', 'delta_mean_zcr', 'delta_std_zcr' ]
scatter_matrix(df_tests[cols], figsize=(12,8))
plt.tight_layout()
plt.show()

In [ ]:
# ==  scatter plot of delta_wer and delta_zcr ==
# plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Noisy ZCR vs WER
sns.scatterplot(x='n_anly.mean_zcr', y='delta_wer', data=df_tests, ax=axes[0], alpha=0.3)
axes[0].set_title('Noisy ZCR vs WER Degradation')
axes[0].set_xlabel("Noisy Audio ZCR")
axes[0].set_ylabel("WER delta")

# Plot 2: Original RMS vs WER
sns.scatterplot(x='c_anly.rms_dbfs_global', y='delta_wer', data=df_tests, ax=axes[1], alpha=0.3)
axes[1].set_title('Original Clean RMS vs WER Degradation')
axes[1].set_xlabel('Clean RMS (dBFS)')
axes[1].set_ylabel('WER Delta')

plt.tight_layout()
plt.show()

In [ ]:
# Plot Noist ZCR vs Delta WER with colored dots for RMS dBFS
sns.scatterplot(
    x='n_anly.mean_zcr',
    y='delta_wer',
    hue='delta_min_zcr',
    palette='viridis',
    data=df_tests,
    alpha=0.3,
    s=100
)

plt.title('Noisy ZCR vs WER Degradation (Colored by delta_min_zcr)')
plt.xlabel('Noisy Audio ZCR')
plt.ylabel('WER Delta (Degradation)')

# Move the legend outside the plot
plt.legend(title='Delta min ZCR', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
df_tests[cols_to_correlate]

In [ ]:
# look only at the distribution of the metrics on the first test (whitenoise)
test_1 = df_tests.loc[df_tests['test_identifier']==1, cols_to_correlate]
test_1


In [ ]:
# look only at the distribution of the metrics on the first test (synth-verb)
test_2 = df_tests.loc[df_tests['test_identifier']==2, cols_to_correlate]
test_2

In [ ]:
# look only at the distribution of the metrics on the first test (real-verb)
test_3 = df_tests.loc[df_tests['test_identifier']==3, cols_to_correlate]
test_3

In [ ]:
# look only at the distribution of the metrics on the first test (dropout)
test_4 = df_tests.loc[df_tests['test_identifier']==4, cols_to_correlate]
test_4

In [ ]:
# look only at the distribution of the metrics on the first test (combined)
test_5 = df_tests.loc[df_tests['test_identifier']==5, cols_to_correlate]
test_5

In [ ]:
# ==  Spearman correlation to delta_wer per test ==
corr_matrix1 = test_1.corr(method='spearman', numeric_only=True)
corr_vals1 = corr_matrix1['delta_wer'].sort_values(ascending=False)

corr_matrix2 = test_2.corr(method='spearman', numeric_only=True)
corr_vals2 = corr_matrix2['delta_wer'].sort_values(ascending=False)

corr_matrix3 = test_3.corr(method='spearman', numeric_only=True)
corr_vals3 = corr_matrix3['delta_wer'].sort_values(ascending=False)

corr_matrix4 = test_4.corr(method='spearman', numeric_only=True)
corr_vals4 = corr_matrix4['delta_wer'].sort_values(ascending=False)

corr_matrix5 = test_5.corr(method='spearman', numeric_only=True)
corr_vals5 = corr_matrix5['delta_wer'].sort_values(ascending=False)

corr_df = pd.concat([corr_vals1, corr_vals2, corr_vals3, corr_vals4, corr_vals5],
                    keys=['test1_whitenoise', 'test2_synt-verb', 'test3_real-verb', 'test4_dropout', 'test5_combined'],
                    axis=1)

corr_df

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(4,3, figsize=(12,8))
axes = axes.flatten()

for i, col in enumerate(df_tests[cols_to_correlate].columns):
    sns.histplot(data=df_tests, x=col, ax=axes[i], kde=True, color="skyblue")
    axes[i].set_title(f"Distribution of {col}", fontsize=12)
    axes[i].set_xlabel(col, fontsize=10)

for ax in axes[len(cols_to_correlate):]:
   ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# plot histograms of all metrics per tests
df_long = pd.melt(
    df_tests,
    id_vars=['test_identifier'],
    value_vars=cols_to_correlate,
    var_name="Metric_Name",
    value_name="Value"
)

sns.set_theme(style="ticks")
sns.set_context("talk")

grid = sns.FacetGrid(
    df_long,
    row='test_identifier',
    col="Metric_Name",
    sharex=False,  # Set to False if metrics have completely different numerical scales
    sharey=True,  # Set to False if some tests have vastly more samples than others
    margin_titles=True,
    height=4,
    aspect=1.3
)

# --- 4. Plot the Histograms ---
grid.map_dataframe(sns.histplot, x="Value", color="royalblue", bins=15)

# Adjust label padding so text doesn't overlap the plots
grid.figure.subplots_adjust(wspace=0.3, hspace=0.4)

# --- 5. Clean up layout ---
grid.tight_layout()
# plt.savefig("./images/high_res_grid.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ==  Perform Train-Test Split on GroupsKFold ==
y = df_tests['delta_wer']
num_cols = ['n_anly.crest_factor_db',
             'n_anly.mean_zcr',
             'n_anly.rms_dbfs_global',
             'c_anly.crest_factor_db',
             'c_anly.mean_zcr',
             'c_anly.rms_dbfs_global',
             'delta_crest_factor',
             'delta_mean_zcr',
             'delta_std_zcr',
             'delta_min_zcr',
             'delta_rms']
X = df_tests[num_cols]

groups = df_tests["idx"].values
outer_gkf = GroupKFold(n_splits=5)

gaussian_cols = [
    'n_anly.crest_factor_db',
    'n_anly.rms_dbfs_global',
    'c_anly.rms_dbfs_global'
]

skewed_cols = [
    'c_anly.mean_zcr',
    'delta_std_zcr',
    'delta_min_zcr'
]

multimodal_cols = [
    'delta_rms',
    'delta_mean_zcr',
    'delta_crest_factor',
    'n_anly.mean_zcr'
]

categorical_cols = [
    'test_identifier'
]

# preprocessor = make_preprocessor(gaussian_cols=gaussian_cols, skewed_cols=skewed_cols, multimodal_cols=multimodal_cols)
selected_models = ['lin_reg', 'rf_reg', 'xgb_reg']
models = {name: all_models[name] for name in selected_models}
param_grids = {name: all_param_grids[name] for name in selected_models}
nested_cv_results = {}

# Nested Cross Validation
for fold, (train_idx, test_idx) in enumerate(outer_gkf.split(X, y, groups=groups)):
    print(f"\n============ OUTER FOLD {fold+1} ============")

    # Generate strict train test split for this fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Slice the groups array to match only the training rows for this fold
    fold_groups = groups[train_idx]

    fold_results = run_multiple_models(
        models,
        param_grids,
        X_train,
        y_train,
        X_test,
        y_test,
        scoring="r2",
        cv=GroupKFold(n_splits=3),
        groups=fold_groups,
        gaussian_cols=gaussian_cols,
        skewed_cols=skewed_cols,
        multimodal_cols=multimodal_cols
    )

    nested_cv_results[f"fold_{fold+1}"] = fold_results



In [ ]:
nested_cv_results['fold_5']['xgb_reg']

In [ ]:
# with open("./results/backup/export_nested_cv_results_2000samples_ML_pass2.pkl", 'wb') as f:
#     pickle.dump(nested_cv_results, f)

---

## Change of stratey - Implement a classification instead of regression


In [ ]:
df_tests["failure"] = (df_tests["delta_wer"] > 0.20).astype(int)
df_tests["degraded"] = (
        (df_tests["delta_wer"] >= -0.20) &
        (df_tests["delta_wer"] < 0.0) &
        (df_tests["delta_wer"] > 0.0) &
        (df_tests["delta_wer"] <= 0.20)).astype(int)
df_tests["nominal"] = (df_tests["delta_wer"] == 0.0).astype(int)
df_tests[["delta_wer", "nominal", "degraded", "failure"]]

In [ ]:
# ==  Perform Train-Test Split on Nested Cross-Validation with GroupsKFold ==
y = df_tests['failure']
features = ['test_identifier',
            'c_anly.crest_factor_db',
            'c_anly.mean_zcr',
            'c_anly.rms_dbfs_global',
            'delta_crest_factor',
            'delta_mean_zcr',
            'delta_std_zcr',
            'delta_min_zcr',
            'delta_rms']
X = df_tests[features]

groups = df_tests["idx"].values
outer_gkf = GroupKFold(n_splits=5)

gaussian_cols = [
    'c_anly.rms_dbfs_global'
]

skewed_cols = [
    'c_anly.mean_zcr',
    'delta_std_zcr',
    'delta_min_zcr'
]

multimodal_cols = [
    'delta_rms',
    'delta_mean_zcr',
    'delta_crest_factor'
]

categorical_cols = [
    'test_identifier'
]

# preprocessor = make_preprocessor(gaussian_cols=gaussian_cols, skewed_cols=skewed_cols, multimodal_cols=multimodal_cols)
selected_models = ['rf_clf', 'xgb_clf', 'svc_rbf']
models = {name: all_models[name] for name in selected_models}
param_grids = {name: all_param_grids[name] for name in selected_models}
nested_cv_results_5 = {}

# Nested Cross Validation
for fold, (train_idx, test_idx) in enumerate(outer_gkf.split(X, y, groups=groups)):
    print(f"\n============ OUTER FOLD {fold+1} ============")

    # Generate strict train test split for this fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Slice the groups array to match only the training rows for this fold
    fold_groups = groups[train_idx]

    fold_results = run_multiple_models(
        models,
        param_grids,
        X_train,
        y_train,
        X_test,
        y_test,
        scoring="f1",
        cv=GroupKFold(n_splits=5),
        groups=fold_groups,
        gaussian_cols=gaussian_cols,
        skewed_cols=skewed_cols,
        multimodal_cols=multimodal_cols,
        categorical_cols=categorical_cols
    )

    nested_cv_results_5[f"fold_{fold+1}"] = fold_results



In [ ]:
# !!! Careful to not overwrite the file with the same name.

# EXPORT
# with open("./results/backup/export_nested_cv_results_2000samples_ML_pass5_clf.pkl", "wb") as f:
#     pickle.dump(nested_cv_results_5, f)

# IMPORT
# with open("./results/backup/export_nested_cv_results_2000samples_ML_pass3_clf.pkl", "rb") as f:
#     nested_cv_results_3 = pickle.load(f)


# with open("./results/backup/export_nested_cv_results_2000samples_ML_pass3_clf.json", "w") as f:
#     json.dump(nested_cv_results_3, f, indent=4)

In [ ]:
nested_cv_results_5

In [ ]:
# == Telemetry parser ==

def parse_nested_cv_telemetry(results_dict):
    """Parses custom nested loop output structure to compute true outer generalization
    and track hyperparameter stabilization
    """
    model_accumulator = {}

    for fold_idx, models_in_fold in results_dict.items():
        for model_name, model_payload in models_in_fold.items():

            if model_name not in model_accumulator:
                model_accumulator[model_name] = {
                    "aucs": [],
                    "precision_scores": [],
                    "recall_scores": [],
                    "f1_scores": [],
                    "confusion_matrices": [],
                    "best_params_per_fold": {}  # stores fold_idx: best_params dict mapping
                }


            test_metrics = model_payload.get("test_metrics", {})
            # print(test_metrics)

            model_accumulator[model_name]["aucs"].append(test_metrics.get("test_auc", 0))
            model_accumulator[model_name]["precision_scores"].append(test_metrics.get("precision", 0))
            model_accumulator[model_name]["recall_scores"].append(test_metrics.get("recall", 0))
            model_accumulator[model_name]["f1_scores"].append(test_metrics.get("f1", 0))

            cm = (
                test_metrics.get("confusion_matrix", None)
            )
            if cm is not None:
                model_accumulator[model_name]["confusion_matrices"].append(test_metrics["confusion_matrix"])
            else:
                model_accumulator[model_name]["missing_confusion_matrix_folds"].append(fold_idx)

            best_params = model_payload.get("best_prams", {})
            model_accumulator[model_name]["best_params_per_fold"][fold_idx] = best_params

    # print(model_accumulator)

    summary_rows = []
    for model_name, telemetry in model_accumulator.items():
        mean_auc = np.mean(telemetry["aucs"]) if telemetry["aucs"] else 0
        mean_precision = np.mean(telemetry["precision_scores"]) if telemetry["precision_scores"] else 0
        mean_recall = np.mean(telemetry["recall_scores"]) if telemetry["recall_scores"] else 0
        mean_f1 = np.mean(telemetry["f1_scores"]) if telemetry["f1_scores"] else 0
        std_f1 = np.std(telemetry["f1_scores"]) if telemetry["f1_scores"] else 0
        summed_cm = np.sum(telemetry["confusion_matrices"], axis=0)
        tn, fp, fn, tp = summed_cm.ravel()
        best_params_per_fold = telemetry["best_params_per_fold"]


        # temporarily convert best params dict payload to tuples for quick uniqueness comparison
        param_tuples = [tuple(sorted(p.items())) for p in telemetry["best_params_per_fold"].values()]
        unique_configs = len(set(param_tuples))
        most_freq_config = max(set(param_tuples), key=param_tuples.count)
        most_freq_config_idx = param_tuples.index(most_freq_config)
        stability_status = "stable" if unique_configs == 1 else f"drifted ({unique_configs} configs)"
        # print(f"model {model_name}")
        # print(f"param_tuples: {param_tuples}")

        summary_rows.append(
            {
                "model_name": model_name,
                "Outer Test F1 (Mean ± Std)": f"{mean_f1:.4f} (± {std_f1:.4f})",
                "ROC-AUC (Mean)": f"{mean_auc:.4f}",
                "Precision (Mean)": f"{mean_precision:.4f}",
                "Recall (Mean)": f"{mean_recall:.4f}",
                "True Negative": int(tn),
                "False Positive": int(fp),
                "False Negative": int(fn),
                "True Positive": int(tp),
                "Missing Confusio Matrix in folds": telemetry.get("missing_confusion_matrix_folds", 0),
                "Parameter Tuning State": stability_status,
                "Dominant Hyperparameter Configuration": json.dumps({e[0]: e[1] for e in most_freq_config}),
            }
        )

    summary_list = [pd.json_normalize(row) for row in summary_rows]
    summary_df = pd.concat(summary_list, ignore_index=True).set_index('model_name')


    raw_hyperparameters_by_model = {
        m: telemetry["best_params_per_fold"] for m, telemetry in model_accumulator.items()
    }

    return summary_df, raw_hyperparameters_by_model



def parse_flatten_cv_telemetry(results_dict):
    """Parses custom nested loop output structure to compute true outer generalization
    and track hyperparameter stabilization
    """

    summary_rows = []

    for model_name, telemetry in results_dict.items():
        cm = telemetry["test_metrics"]["confusion_matrix"]
        tn, fp, fn, tp = cm.ravel()
        best_params = telemetry["best_prams"]

        summary_rows.append(
            {
                "model_name": model_name,
                "F1 score": f"{telemetry["test_metrics"]["f1"]:.4f}",
                "ROC-AUC": f"{telemetry["test_metrics"]["test_auc"]:.4f}",
                "Precision": f"{telemetry["test_metrics"]["precision"]:.4f}",
                "Recall": f"{telemetry["test_metrics"]["recall"]:.4f}",
                "True Negative": int(tn),
                "False Positive": int(fp),
                "False Negative": int(fn),
                "True Positive": int(tp),
                "Best parameters": json.dumps(best_params),            }
        )

    summary_list = [pd.json_normalize(row) for row in summary_rows]
    summary_df = pd.concat(summary_list, ignore_index=True).set_index('model_name')


    raw_hyperparameters_by_model = {
        m: telemetry["best_prams"] for m, telemetry in results_dict.items()
    }

    return summary_df, raw_hyperparameters_by_model



In [ ]:
summary_df, raw_hyperparams_by_model = parse_nested_cv_telemetry(nested_cv_results_5)

In [ ]:
summary_df

In [ ]:
raw_hyperparams_by_model

In [ ]:
# ==  Perform Train-Test Split on flatten GroupsKFold ==
y = df_tests['failure']
features = ['test_identifier',
            'c_anly.crest_factor_db',
            'c_anly.mean_zcr',
            'c_anly.rms_dbfs_global',
            'delta_crest_factor',
            'delta_mean_zcr',
            'delta_std_zcr',
            'delta_min_zcr',
            'delta_rms']
X = df_tests[features]
print(f"X shape: {X.shape}")
groups = df_tests["idx"].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)

gaussian_cols = [
    'c_anly.rms_dbfs_global'
]

skewed_cols = [
    'c_anly.mean_zcr',
    'delta_std_zcr',
    'delta_min_zcr'
]

multimodal_cols = [
    'delta_rms',
    'delta_mean_zcr',
    'delta_crest_factor'
]

categorical_cols = [
    'test_identifier'
]

# preprocessor = make_preprocessor(gaussian_cols=gaussian_cols, skewed_cols=skewed_cols, multimodal_cols=multimodal_cols)
selected_models = ['rf_clf', 'xgb_clf', 'svc_rbf']
models = {name: all_models[name] for name in selected_models}
param_grids = {name: all_param_grids[name] for name in selected_models}

# Generate strict train test split
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

print(f"groups length: {len(groups)}")
print(f"groups_train length: {len(groups_train)}")
print(f"groups_test length: {len(groups_test)}")

nested_cv_results_3_flatten = run_multiple_models(
    models,
    param_grids,
    X_train,
    y_train,
    X_test,
    y_test,
    scoring="f1",
    cv=GroupKFold(n_splits=5),
    groups=groups_train,
    gaussian_cols=gaussian_cols,
    skewed_cols=skewed_cols,
    multimodal_cols=multimodal_cols,
    categorical_cols=categorical_cols
)

In [ ]:
# # export data
# with open("./results/backup/export_nested_cv_results_2000samples_ML_pass5_clf_flatten.pkl", "wb") as f:
#     pickle.dump(nested_cv_results_3_flatten, f)

In [ ]:
nested_cv_results_3_flatten

In [ ]:
summary_df_flatten, raw_hyperparams_by_model_flatten = parse_flatten_cv_telemetry(nested_cv_results_3_flatten)

In [ ]:
summary_df_flatten

---

## Compare results

In [ ]:
summary_df_flatten

In [ ]:
summary_df

In [ ]:
flatten_rf_model = nested_cv_results_3_flatten['rf_clf']['best_model'].steps[-1][1]
flatten_rf_model